In [62]:
import pandas as pd

In [63]:
from insurance_pricing import data, config

In [64]:
df_raw = data.load_raw()

In [65]:
df = data.add_derived_features(df_raw)
df

,ID,Date_start_contract,Date_last_renewal,Date_next_renewal,Date_birth,Date_driving_licence,Distribution_channel,Seniority,Policies_in_force,Max_policies,...,Value_vehicle,N_doors,Type_fuel,Length,Weight,exposure,driver_age,licence_years,vehicle_age,renewal_year
0,1,2015-11-05,2015-11-05,2016-11-05,1956-04-15,1976-03-20,0,4,1,2,...,7068.00,0,P,NaN,190,1.000000,59.597260,39.654795,11,2015
1,1,2015-11-05,2016-11-05,2017-11-05,1956-04-15,1976-03-20,0,4,1,2,...,7068.00,0,P,NaN,190,1.000000,60.600000,40.657534,12,2016
2,1,2015-11-05,2017-11-05,2018-11-05,1956-04-15,1976-03-20,0,4,2,2,...,7068.00,0,P,NaN,190,1.000000,61.600000,41.657534,13,2017
3,1,2015-11-05,2018-11-05,2019-11-05,1956-04-15,1976-03-20,0,4,2,2,...,7068.00,0,P,NaN,190,0.153425,62.600000,42.657534,14,2018
4,2,2017-09-26,2017-09-26,2018-09-26,1956-04-15,1976-03-20,0,4,2,2,...,7068.00,0,P,NaN,190,1.000000,61.490411,41.547945,13,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105550,53498,2018-07-30,2018-07-30,2019-07-30,1981-07-25,2007-02-14,0,1,1,1,...,24320.00,5,D,4.740,1480,0.421918,37.038356,11.463014,18,2018
105551,53499,2018-08-16,2018-08-16,2019-08-16,1976-12-08,2017-11-29,0,1,1,1,...,30861.97,5,P,4.650,1440,0.375342,41.715068,0.712329,5,2018
105552,53500,2018-11-21,2018-11-21,2019-11-21,1974-04-01,2011-10-05,0,1,1,1,...,7800.00,5,P,3.495,830,0.109589,44.671233,7.134247,19,2018
105553,53501,2018-11-21,2018-11-21,2019-11-21,1946-09-15,1982-02-02,0,1,1,1,...,16610.00,5,D,4.555,1399,0.109589,72.232877,36.824658,14,2018


In [66]:
df_audit = df.copy()

grouped = df_audit.groupby(config.ID_COL)

n_unique_per_id = grouped.nunique()
rows_per_id = grouped.size()

n_unique_multi = n_unique_per_id[rows_per_id >= 2]

constant_share = (n_unique_multi == 1).mean().sort_values(ascending=False)
constant_share

Date_start_contract     1.000000
Date_birth              1.000000
Weight                  1.000000
Date_driving_licence    1.000000
Distribution_channel    1.000000
Seniority               1.000000
Max_products            1.000000
Max_policies            1.000000
R_Claims_history        1.000000
Payment                 1.000000
N_claims_history        1.000000
Value_vehicle           1.000000
Cylinder_capacity       1.000000
Power                   1.000000
Year_matriculation      1.000000
Second_driver           1.000000
Area                    1.000000
Type_risk               1.000000
N_doors                 1.000000
Type_fuel               0.982839
Length                  0.898612
Policies_in_force       0.755946
Cost_claims_year        0.645780
N_claims_year           0.645294
Lapse                   0.629681
Date_lapse              0.348632
exposure                0.226893
Premium                 0.039729
Date_last_renewal       0.000000
Date_next_renewal       0.000000
driver_age

In [67]:
corr_with_target = (
    df_audit.corr(numeric_only=True)[config.N_CLAIMS_COL]
    .drop(config.N_CLAIMS_COL)
    .sort_values(key=abs, ascending=False)
)
corr_with_target


R_Claims_history        0.565950
N_claims_history        0.392514
Cost_claims_year        0.199022
renewal_year           -0.193820
Policies_in_force       0.176735
Max_policies            0.158704
exposure                0.150208
Payment                 0.111551
Premium                 0.092688
Cylinder_capacity       0.081607
Length                  0.078105
Weight                  0.072789
Value_vehicle           0.068106
Power                   0.063777
licence_years          -0.050956
N_doors                 0.050852
Seniority              -0.043957
driver_age             -0.041258
Area                    0.032624
Second_driver           0.032343
Distribution_channel    0.028695
vehicle_age            -0.026840
Type_risk               0.022593
Lapse                  -0.011383
ID                      0.007870
Year_matriculation      0.001116
Max_products           -0.000132
Name: N_claims_year, dtype: float64

In [68]:

(df_audit["N_claims_history"] >= df_audit[config.N_CLAIMS_COL]).all()

np.True_

In [69]:
data.anomaly_report(df)

driver_age_below_18        0
licence_years_negative    31
licence_before_18          0
vehicle_age_negative       0
missing_dates              0
dtype: int64

In [70]:
df = data.fix_anomalies(df)
data.anomaly_report(df)

driver_age_below_18       0
licence_years_negative    0
licence_before_18         0
vehicle_age_negative      0
missing_dates             0
dtype: int64

In [71]:
df = data.drop_unused_columns(df)
df

,ID,Date_last_renewal,Distribution_channel,Payment,Cost_claims_year,N_claims_year,Type_risk,Area,Second_driver,Power,...,Value_vehicle,N_doors,Type_fuel,Length,Weight,exposure,driver_age,licence_years,vehicle_age,renewal_year
0,1,2015-11-05,0,0,0.0,0,1,0,0,80,...,7068.00,0,P,NaN,190,1.000000,59.597260,39.654795,11,2015
1,1,2016-11-05,0,0,0.0,0,1,0,0,80,...,7068.00,0,P,NaN,190,1.000000,60.600000,40.657534,12,2016
2,1,2017-11-05,0,0,0.0,0,1,0,0,80,...,7068.00,0,P,NaN,190,1.000000,61.600000,41.657534,13,2017
3,1,2018-11-05,0,0,0.0,0,1,0,0,80,...,7068.00,0,P,NaN,190,0.153425,62.600000,42.657534,14,2018
4,2,2017-09-26,0,1,0.0,0,1,0,0,80,...,7068.00,0,P,NaN,190,1.000000,61.490411,41.547945,13,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105550,53498,2018-07-30,0,0,0.0,0,3,0,0,110,...,24320.00,5,D,4.740,1480,0.421918,37.038356,11.463014,18,2018
105551,53499,2018-08-16,0,0,0.0,0,3,0,0,129,...,30861.97,5,P,4.650,1440,0.375342,41.715068,0.712329,5,2018
105552,53500,2018-11-21,0,1,0.0,0,3,0,0,55,...,7800.00,5,P,3.495,830,0.109589,44.671233,7.134247,19,2018
105553,53501,2018-11-21,0,0,0.0,0,2,0,0,90,...,16610.00,5,D,4.555,1399,0.109589,72.232877,36.824658,14,2018


In [72]:
exposure_claims_table = df.groupby(config.RENEWAL_YEAR_COL).agg(num_of_rows = (config.ID_COL, "size"), 
                                        mean_exposure = (config.EXPOSURE_COL, "mean"), 
                                        mean_claims = (config.N_CLAIMS_COL, "mean"),
                                        sum_claims = (config.N_CLAIMS_COL, "sum"),
                                        sum_exposure = (config.EXPOSURE_COL, "sum"))


exposure_claims_table

,num_of_rows,mean_exposure,mean_claims,sum_claims,sum_exposure
renewal_year,,,,,
2015,4559,0.977684,0.710682,3240,4457.260274
2016,31428,0.978844,0.663803,20862,30763.112329
2017,33753,0.980468,0.367493,12404,33093.731507
2018,35815,0.522859,0.143962,5156,18726.208219


In [73]:
exposure_claims_table["frequency"] = exposure_claims_table["sum_claims"] / exposure_claims_table["sum_exposure"]

exposure_claims_table["frequency"]

renewal_year
2015    0.726904
2016    0.678150
2017    0.374814
2018    0.275336
Name: frequency, dtype: float64

In [74]:
df_2018 = df[df[config.RENEWAL_YEAR_COL] == 2018].copy()

df_2018["exposure_bin"] = pd.cut(df_2018[config.EXPOSURE_COL], bins=[0, 0.25, 0.5, 0.75, 1.0])

exposure_bins_table = df_2018.groupby("exposure_bin", observed=True).agg(
    num_of_rows=(config.ID_COL, "size"),
    mean_claims=(config.N_CLAIMS_COL, "mean"),
)

exposure_bins_table # mean_claims is scaling with exposure

,num_of_rows,mean_claims
exposure_bin,,
"(0.0, 0.25]",7060,0.010198
"(0.25, 0.5]",10060,0.083002
"(0.5, 0.75]",10028,0.181990
"(0.75, 1.0]",8667,0.279682


In [75]:
data.save_processed(df)

df.equals(data.build_dataset())

True